*  DSC550-T302 Data Mining (2265-1)
*  9.2 Exercise: Best Model Selection and Hyperparameter Tuning
*  David Jorgensen

# 9.2 Exercise: Best Model Selection and Hyperparameter Tuning

# Instructions

In this exercise, you will work with the `./Loan_Train.csv` dataset.

- Import the dataset and ensure that it loaded properly.
- Prepare the data for modeling by performing the following steps:
- Drop the column “Loan_ID.”
- Drop any rows with missing data.
- Convert the categorical features into dummy variables.
- Split the data into a training and test set, where the “Loan_Status” column is the target.
- Create a pipeline with a min-max scaler and a KNN classifier (see section 15.3 in the Machine Learning with Python Cookbook).
- Fit a default KNN classifier to the data with this pipeline. Report the model accuracy on the test set. Note: Fitting a pipeline model works just like fitting a regular model.
- Create a search space for your KNN classifier where your “n_neighbors” parameter varies from 1 to 10. (see section 15.3 in the Machine Learning with Python Cookbook).
- Fit a grid search with your pipeline, search space, and 5-fold cross-validation to find the best value for the “n_neighbors” parameter.
- Find the accuracy of the grid search best model on the test set. Note: It is possible that this will not be an improvement over the default model, but likely it will be.
- Now, repeat steps 6 and 7 with the same pipeline, but expand your search space to include logistic regression and random forest models with the hyperparameter values in section 12.3 of the Machine Learning with Python Cookbook.
- What are the best model and hyperparameters found in the grid search? Find the accuracy of this model on the test set.
- Summarize your results.


Submission Instructions:

- Submit your code, output, and answers. Comment all your code and answer any questions that are asked in the instructions. It is perfectly fine to answer a question by displaying output from your code, but make sure you are displaying the appropriate output to answer the question.
- Make sure to comment all your code as you are documenting your steps, process, and analysis.

# Assignment

## Steps 1–2: Load and Prepare Loan Data

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import numpy as np
from IPython.display import HTML

def wrap_table(df, cols_per_chunk=5):
    """Split wide DataFrame into stacked sub-tables for narrow paper printing.
    Each sub-table shows the index plus cols_per_chunk columns.
    I'm including this because I got docked points on previous assignments
    for clipped tables in the PDF export. If I get docked for including
    'extra code' because I'm trying to mitigate another issue, I will riot. 
    """
    cols = list(df.columns)
    chunks = [cols[i:i + cols_per_chunk] for i in range(0, len(cols), cols_per_chunk)]

    t  = 'width:100%; border-collapse:collapse; font-size:0.85em; margin-bottom:1.2em;'
    th = ('padding:3px 8px; text-align:left; background:#f2f2f2; '
          'border-bottom:2px solid #444; border-right:1px solid #ccc; '
          'overflow-wrap:anywhere; word-break:break-word;')
    td = ('padding:3px 8px; text-align:left; '
          'border-bottom:1px solid #ddd; border-right:1px solid #ccc; '
          'overflow-wrap:anywhere; word-break:break-word;')

    parts = []
    idx_name = df.index.name or ''
    for chunk in chunks:
        parts.append(f'<table style="{t}"><thead><tr>')
        parts.append(f'<th style="{th}">{idx_name}</th>')
        for col in chunk:
            parts.append(f'<th style="{th}">{col}</th>')
        parts.append('</tr></thead><tbody>')
        for i, (idx, row) in enumerate(df.iterrows()):
            bg = '#fafafa' if i % 2 == 0 else '#ffffff'
            parts.append(f'<tr style="background:{bg}">')
            parts.append(f'<td style="{td}">{idx}</td>')
            for col in chunk:
                parts.append(f'<td style="{td}">{row[col]}</td>')
            parts.append('</tr>')
        parts.append('</tbody></table>')

    return HTML(''.join(parts))

# Step 1: Load loan dataset and confirm it loaded correctly
df = pd.read_csv('Loan_Train.csv')
print('Shape:', df.shape)
wrap_table(df.head())

Shape: (614, 13)


,Loan_ID,Gender,Married,Dependents,Education
0,LP001002,Male,No,0,Graduate
1,LP001003,Male,Yes,1,Graduate
2,LP001005,Male,Yes,0,Graduate
3,LP001006,Male,Yes,0,Not Graduate
4,LP001008,Male,No,0,Graduate
,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term
0,No,5849,0.0,nan,360.0
1,No,4583,1508.0,128.0,360.0
2,Yes,3000,0.0,66.0,360.0
3,No,2583,2358.0,120.0,360.0


In [2]:
# Step 2a: Drop Loan_ID — not a predictive feature
df = df.drop(columns=['Loan_ID'])

# Step 2b: Drop rows with any missing values
df = df.dropna()
print('Shape after cleaning:', df.shape)
print('Remaining nulls:', df.isnull().sum().sum())

Shape after cleaning: (480, 12)
Remaining nulls: 0


In [3]:
# Step 2c: Convert categorical features to dummy variables
# drop_first=True removes one dummy per category to avoid multicollinearity
df_enc = pd.get_dummies(df, drop_first=True)
print('Shape after encoding:', df_enc.shape)
wrap_table(df_enc.head(3))

Shape after encoding: (480, 15)


,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
1,4583,1508.0,128.0,360.0,1.0
2,3000,0.0,66.0,360.0,1.0
3,2583,2358.0,120.0,360.0,1.0
,Gender_Male,Married_Yes,Dependents_1,Dependents_2,Dependents_3+
1,True,True,True,False,False
2,True,True,False,False,False
3,True,True,False,False,False
,Education_Not Graduate,Self_Employed_Yes,Property_Area_Semiurban,Property_Area_Urban,Loan_Status_Y
1,False,False,False,False,False
2,False,True,False,True,True


## Step 3: Train/Test Split

In [4]:
# Step 3: Separate features from target (Loan_Status_Y: 1=approved, 0=rejected)
X = df_enc.drop(columns=['Loan_Status_Y'])
y = df_enc['Loan_Status_Y'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('Train size:', X_train.shape, '  Test size:', X_test.shape)
print('Target distribution (train):', y_train.value_counts().to_dict())

Train size: (384, 14)   Test size: (96, 14)
Target distribution (train): {1: 264, 0: 120}


## Steps 4–5: Default KNN Pipeline

In [5]:
# Step 4: Build pipeline with MinMaxScaler and KNN
# Name the classifier step 'classifier' so we can swap models later in the search space
pipe = Pipeline([
    ('scaler', MinMaxScaler()),
    ('classifier', KNeighborsClassifier())
])

# Step 5: Fit default KNN pipeline (k=5 by default) and report test accuracy
pipe.fit(X_train, y_train)
default_acc = accuracy_score(y_test, pipe.predict(X_test))
print(f'Default KNN (k=5) accuracy on test set: {default_acc:.4f}')

Default KNN (k=5) accuracy on test set: 0.7812


## Steps 6–8: KNN Grid Search (n_neighbors 1–10)

In [6]:
# Step 6: Define search space — vary n_neighbors from 1 to 10
knn_search_space = [{'classifier__n_neighbors': range(1, 11)}]

# Step 7: Grid search with 5-fold CV to find optimal k
knn_gs = GridSearchCV(pipe, knn_search_space, cv=5, n_jobs=-1)
knn_gs.fit(X_train, y_train)

best_k = knn_gs.best_params_['classifier__n_neighbors']
print(f'Best n_neighbors: {best_k}')
print(f'Best CV score:    {knn_gs.best_score_:.4f}')

# Step 8: Accuracy of best KNN model on test set
knn_best_acc = accuracy_score(y_test, knn_gs.predict(X_test))
print(f'Best KNN accuracy on test set: {knn_best_acc:.4f}')

Best n_neighbors: 3
Best CV score:    0.7423
Best KNN accuracy on test set: 0.7917


## Steps 9–10: Multi-Model Grid Search (KNN + Logistic Regression + Random Forest)

In [7]:
# Step 9: Expand search space to include Logistic Regression and Random Forest
# Hyperparameter values from section 12.3 of Machine Learning with Python Cookbook
multi_search_space = [
    # KNN: n_neighbors 1–10
    {
        'classifier': [KNeighborsClassifier()],
        'classifier__n_neighbors': range(1, 11),
    },
    # Logistic Regression: vary regularization strength and penalty type
    # solver='liblinear' supports both l1 and l2 penalties
    {
        'classifier': [LogisticRegression(max_iter=1000, solver='liblinear')],
        'classifier__penalty': ['l1', 'l2'],
        'classifier__C': np.logspace(0, 4, 10),
    },
    # Random Forest: vary number of trees and features considered per split
    {
        'classifier': [RandomForestClassifier(random_state=42)],
        'classifier__n_estimators': [10, 100, 1000],
        'classifier__max_features': [1, 2, 3],
    },
]

# Grid search with 5-fold CV across all three model families
multi_gs = GridSearchCV(pipe, multi_search_space, cv=5, n_jobs=-1)
multi_gs.fit(X_train, y_train)

# Step 10: Report best model and hyperparameters
best_clf = multi_gs.best_params_['classifier']
best_other = {k: v for k, v in multi_gs.best_params_.items() if k != 'classifier'}
print(f'Best model:  {type(best_clf).__name__}')
print(f'Best params: {best_other}')
print(f'Best CV score: {multi_gs.best_score_:.4f}')

# Accuracy on test set
multi_best_acc = accuracy_score(y_test, multi_gs.predict(X_test))
print(f'Best multi-model accuracy on test set: {multi_best_acc:.4f}')

Best model:  LogisticRegression
Best params: {'classifier__C': np.float64(7.742636826811269), 'classifier__penalty': 'l1'}
Best CV score: 0.8100
Best multi-model accuracy on test set: 0.8229


## Step 11: Summary

In [8]:
# Step 11: Summarize results across all three models
results = pd.DataFrame({
    'Model': [
        'Default KNN (k=5)',
        f'Best KNN (k={knn_gs.best_params_["classifier__n_neighbors"]})',
        f'Best Multi-Model ({type(multi_gs.best_params_["classifier"]).__name__})'
    ],
    'Accuracy': [default_acc, knn_best_acc, multi_best_acc]
})
results['Accuracy'] = results['Accuracy'].map('{:.4f}'.format)
wrap_table(results)

,Model,Accuracy
0,Default KNN (k=5),0.7812
1,Best KNN (k=3),0.7917
2,Best Multi-Model (LogisticRegression),0.8229


### Findings

The raw dataset contained **614 rows and 13 columns**. After dropping `Loan_ID` and removing the 134 rows with any missing value (primarily in `LoanAmount`, `Loan_Amount_Term`, `Credit_History`, and `Gender`), **480 rows** remained. Converting the categorical features to dummy variables expanded the feature space to **14 predictors** (plus the target). The training set class balance was 264 approved vs. 120 rejected (~69% approval rate), reflecting a moderately imbalanced binary classification problem.

**Default KNN (k=5):** The baseline MinMaxScaler + KNN pipeline achieved **78.12% accuracy** on the 96-row test set. KNN requires scaling because it computes Euclidean distances; without MinMaxScaler, high-variance numeric features like `ApplicantIncome` would dominate the distance calculation.

**KNN Grid Search (n_neighbors 1–10):** The 5-fold cross-validation grid search selected **k=3** as the optimal number of neighbors (CV score 74.23%). Despite k=3 having a lower CV score than the default k=5, it achieved **79.17% accuracy** on the test set, which is a modest improvement of 1.05 percentage points over the default. The divergence between CV score and test accuracy suggests some variance in this small dataset (384 training samples).

**Multi-Model Grid Search:** Expanding the search to include Logistic Regression (C from logspace(0, 4, 10), l1/l2 penalty) and Random Forest (10/100/1000 trees, 1/2/3 max features) identified **Logistic Regression with C≈7.74 and L1 penalty** as the best model (CV score 81.00%, test accuracy **82.29%**). This is an improvement of 4.17 percentage points over the default KNN and 3.13 points over the best KNN. L1 regularization performs implicit feature selection by driving some coefficients to zero, which is useful here given that some dummy-encoded features (e.g., rare dependent counts) may carry little signal. The relatively high C value (weak regularization) indicates the model benefited from fitting the training data more closely rather than applying heavy shrinkage.

**Takeaway:** Hyperparameter tuning improved accuracy at each step. The largest gain came from switching model families entirely. Logistic Regression outperformed KNN by ~4 percentage points. For a loan approval classifier, the most costly error is approving a bad loan (false negative on default risk). With an 82% accurate model on a dataset this size, further gains would likely require additional feature engineering or more training data rather than continued hyperparameter search.